# 08 — YOLOv8-seg + NWD box loss (V15, training only)

**Training-only**, Kaggle-self-contained. New experiment after the MedSAM Phase-0 eval
(`results/version14_results.csv`). This is the **V15** training run → save its `results.csv` as
`results/version15_results.csv`.

## Hypothesis
Both closed research lines (two-stage refine in `src/04`–`src/06`, MedSAM mask swap in `src/07`)
proved the same thing: with a **tight box**, the small Caries classes can jump (+0.11–0.22 at the
GT-box oracle). The blocker is that V6's small boxes are **loose**, and the reason is that
**IoU/CIoU is hyper-sensitive to a few-pixel shift on tiny objects** → unstable regression gradient
→ the small boxes never tighten. NWD (Normalized Gaussian Wasserstein Distance) models each box as a
2-D Gaussian and is smooth under small localization error. Blending it into the box regression loss
should tighten the small boxes — the lever both prior lines were waiting on.

## Single variable vs V6
The **only** change is the box regression loss (section 7). Model (`yolov8s-seg`), input (full image,
`imgsz=768`), augmentation, and data are the **clean V6 baseline**. So the in-training Mask mAP50-95
here is **directly comparable to V6 ≈0.234** (unlike V13's tiled-val number).

## How to read the result
1. Headline: best Mask `metrics/mAP50-95(M)` vs V6 0.234 (noise band ≈0.003).
2. **Leading indicator of "did the boxes get tighter":** re-run `src/05` Phase-1a with this run's
   `best.pt` and compare small-Caries **localization recall@IoU0.5** vs V6 — the most direct
   box-quality proxy; it should move before the aggregate mAP does.
3. Per-class: watch Caries 1/2/3/5 (the supported ones) AND confirm the large classes
   (Abrasion/Crown/Filling) do **not** regress.

## Knobs (section 6)
`NWD_IOU_RATIO` (λ, weight on CIoU) and `NWD_CONSTANT` (C, the normalization scale). **C is the key
hyperparameter** and lives in stride-normalized units — plan a small sweep `{3, 5, 8}`.

## 1. Environment Setup

Imports and a fixed seed for reproducibility.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Current working directory:", os.getcwd())
print("Kaggle input directory exists:", Path("/kaggle/input").exists())
print("Kaggle working directory exists:", Path("/kaggle/working").exists())

In [ ]:
# Confirm the accelerator is available.
!nvidia-smi

## 2. Install and Import Ultralytics

Installs Ultralytics if it is not already present, then imports YOLO. NWD is added by patching Ultralytics' loss (section 7), not by a fork.

In [ ]:
try:
    import ultralytics
    print("Ultralytics is already installed.")
except ImportError:
    print("Installing Ultralytics...")
    !pip install -q ultralytics
    import ultralytics

In [ ]:
os.environ["YOLO_VERBOSE"] = "False"

import torch
from ultralytics import YOLO
from ultralytics.utils import LOGGER

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("Ultralytics:", ultralytics.__version__)

## 3. Locate the Dataset YAML

Searches for `yolo_seg_train.yaml` under `/kaggle/input` (the original **full-image** AlphaDent dataset — this run is NOT tiled).

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
yaml_candidates = list(INPUT_ROOT.rglob("yolo_seg_train.yaml"))

print(f"Number of yolo_seg_train.yaml files found: {len(yaml_candidates)}")
for i, p in enumerate(yaml_candidates):
    print(f"[{i}] {p}")

if len(yaml_candidates) == 0:
    raise FileNotFoundError(
        "No yolo_seg_train.yaml found under /kaggle/input. "
        "Check that the AlphaDent dataset is added to this notebook."
    )

YAML_INDEX = 0
DATA_YAML = yaml_candidates[YAML_INDEX]
print("\nSelected DATA_YAML:", DATA_YAML)
print("Dataset root:", DATA_YAML.parent)

## 4. Inspect Dataset Metadata

Reads the YAML and resolves the class names / count.

In [ ]:
with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_yaml_dict = yaml.safe_load(f)

print("YAML content:")
for k, v in data_yaml_dict.items():
    print(f"{k}: {v}")

names = data_yaml_dict.get("names", None)
if names is None:
    raise ValueError("The YAML file does not contain the 'names' field.")
num_classes = len(names)
print("\nClass names:", names)
print("Number of classes:", num_classes)

## 5. Basic Path Check

Resolves the train / val image folders to absolute paths.

In [ ]:
dataset_root = DATA_YAML.parent
yaml_path_value = data_yaml_dict.get("path", None)

if yaml_path_value is not None:
    yaml_root = Path(yaml_path_value)
    if not yaml_root.is_absolute():
        yaml_root = dataset_root / yaml_root
else:
    yaml_root = dataset_root

print("Resolved YAML root:", yaml_root)

def resolve_split_path(split_value):
    if split_value is None:
        return None
    split_path = Path(split_value)
    if split_path.is_absolute():
        return split_path
    candidate_1 = yaml_root / split_path
    if candidate_1.exists():
        return candidate_1
    candidate_2 = dataset_root / split_path
    if candidate_2.exists():
        return candidate_2
    return candidate_1

train_path = resolve_split_path(data_yaml_dict.get("train", None))
val_path = resolve_split_path(data_yaml_dict.get("val", data_yaml_dict.get("valid", None)))

print("Train path:", train_path, "| exists:", train_path.exists() if train_path else None)
print("Val path:", val_path, "| exists:", val_path.exists() if val_path else None)

if train_path is None or not train_path.exists():
    raise FileNotFoundError("Train path does not exist. Please check the YAML file.")
if val_path is None or not val_path.exists():
    raise FileNotFoundError("Validation path does not exist. Please check the YAML file.")

In [ ]:
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images(folder):
    if folder is None or not folder.exists():
        return 0
    return sum(1 for p in folder.rglob("*") if p.suffix.lower() in image_exts)

split_counts = {"train": count_images(train_path), "val": count_images(val_path)}
pd.DataFrame([{"split": k, "num_images": v} for k, v in split_counts.items()])

## 6. Training Configuration (V15 — NWD)

Full-image **V6 baseline** plus the NWD knobs. Edit only this cell to sweep the NWD hyperparameters.

In [ ]:
# =========================================================
# V15 Training configuration  (NWD box loss — single variable vs V6)
# =========================================================
# Versioning note: results/version14_results.csv is the MedSAM Phase-0 EVAL table.
# This NWD run is the next *training* experiment -> V15 -> results/version15_results.csv.
#
# Baseline = V6 (current best, ~0.234 Mask mAP50-95): stock yolov8s-seg, imgsz=768, clean
# augmentation, no oversampling, full-image input. The ONLY new variable is the NWD-blended
# box regression loss applied in section 7.

MODEL_NAME = "yolov8s-seg.pt"
MODEL_TAG = "yolov8s_seg"
IMG_SIZE = 768

EPOCHS = 120
BATCH_SIZE = 16
PATIENCE = 30
DEVICE = 0
WORKERS = 8

# --- clean V6 augmentation (kept identical so NWD is the only change) ---
FLIPLR = 0.5
MOSAIC = 1.0
CLOSE_MOSAIC = 10
MIXUP = 0.0
COPY_PASTE = 0.0

# --- NWD knobs ---
# NWD_ENABLE       : master switch. False -> pure V6 CIoU baseline (use to sanity-check parity).
# NWD_IOU_RATIO (L): weight on the CIoU term; NWD weight = 1 - L. 1.0 = baseline, 0.5 = even blend.
# NWD_CONSTANT  (C): normalization scale of the Wasserstein distance, in STRIDE-NORMALIZED units.
#                    THIS IS THE KEY HYPERPARAMETER. Too large -> NWD ~ 1 (no effect); too small ->
#                    NWD saturates to 0 (no gradient). Recommended first sweep: {3, 5, 8}.
NWD_ENABLE = True
NWD_IOU_RATIO = 0.5
NWD_CONSTANT = 5.0

PROJECT_DIR = Path("/kaggle/working/yolo_runs")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

_c_tag = str(NWD_CONSTANT).replace(".", "p")
RUN_NAME = f"v15_nwd_r{int(round(NWD_IOU_RATIO * 100))}_c{_c_tag}_{MODEL_TAG}_imgsz{IMG_SIZE}"
print("RUN_NAME:", RUN_NAME)
print(f"NWD_ENABLE={NWD_ENABLE}, NWD_IOU_RATIO={NWD_IOU_RATIO}, NWD_CONSTANT={NWD_CONSTANT}")

## 7. NWD loss patch

Monkey-patches `ultralytics.utils.loss.BboxLoss.forward` at the **class** level so the loss object
the trainer builds later (inside `v8SegmentationLoss`) picks up the change. The box regression loss
becomes:

```
box_loss = L * (1 - CIoU) + (1 - L) * (1 - NWD)
```

Large boxes keep being driven by CIoU; small boxes get the stable NWD signal. The DFL term is left
exactly as stock. This is the **only** change vs V6.

> A stronger (more invasive) variant also blends NWD into the `TaskAlignedAssigner` metric — left as
> a possible follow-up so this run stays a clean single variable.

In [ ]:
# =========================================================
# V15: NWD (Normalized Gaussian Wasserstein Distance) box-loss patch
# =========================================================
import torch
import ultralytics.utils.loss as _ul_loss
from ultralytics.utils.metrics import bbox_iou as _bbox_iou

# Keep a handle to the original so the patch is reversible / auditable AND so we can
# reuse its (version-specific) DFL computation unchanged.
if "_ORIG_BBOX_FORWARD" not in globals():
    _ORIG_BBOX_FORWARD = _ul_loss.BboxLoss.forward


def _nwd_similarity(pred, target, constant, eps=1e-7):
    # Normalized Gaussian Wasserstein similarity in [0, 1] for xyxy box pairs.
    # pred/target: (N, 4) in the SAME (stride-normalized) units BboxLoss receives.
    cxp = (pred[:, 0] + pred[:, 2]) * 0.5
    cyp = (pred[:, 1] + pred[:, 3]) * 0.5
    wp = (pred[:, 2] - pred[:, 0]).clamp(min=0)
    hp = (pred[:, 3] - pred[:, 1]).clamp(min=0)
    cxt = (target[:, 0] + target[:, 2]) * 0.5
    cyt = (target[:, 1] + target[:, 3]) * 0.5
    wt = (target[:, 2] - target[:, 0]).clamp(min=0)
    ht = (target[:, 3] - target[:, 1]).clamp(min=0)
    center = (cxp - cxt) ** 2 + (cyp - cyt) ** 2
    wh = ((wp - wt) ** 2 + (hp - ht) ** 2) * 0.25
    w2 = center + wh                       # squared Gaussian Wasserstein distance
    return torch.exp(-torch.sqrt(w2 + eps) / constant)


def _make_nwd_forward(iou_ratio, constant):
    # NOTE: BboxLoss.forward's signature has changed across Ultralytics versions
    # (older = 7 positional args ending in fg_mask; newer >=8.3 appends imgsz, stride).
    # We absorb any trailing args with *extra and recompute ONLY the regression term,
    # then delegate the DFL term to the stock forward so this stays version-robust.
    def forward(self, pred_dist, pred_bboxes, anchor_points, target_bboxes,
                target_scores, target_scores_sum, fg_mask, *extra):
        # --- regression term: blend CIoU with NWD on the foreground boxes ---
        weight = target_scores.sum(-1)[fg_mask].unsqueeze(-1)
        p_fg = pred_bboxes[fg_mask]
        t_fg = target_bboxes[fg_mask]

        iou = _bbox_iou(p_fg, t_fg, xywh=False, CIoU=True)            # (N, 1)
        nwd = _nwd_similarity(p_fg, t_fg, constant).unsqueeze(-1)     # (N, 1)
        blended = iou_ratio * (1.0 - iou) + (1.0 - iou_ratio) * (1.0 - nwd)
        loss_iou = (blended * weight).sum() / target_scores_sum

        # --- DFL term: reuse the stock implementation (its loss_iou is discarded) ---
        _, loss_dfl = _ORIG_BBOX_FORWARD(
            self, pred_dist, pred_bboxes, anchor_points, target_bboxes,
            target_scores, target_scores_sum, fg_mask, *extra
        )
        return loss_iou, loss_dfl
    return forward


if NWD_ENABLE and NWD_IOU_RATIO < 1.0:
    _ul_loss.BboxLoss.forward = _make_nwd_forward(NWD_IOU_RATIO, NWD_CONSTANT)
    print(f"NWD patch APPLIED: box_loss = {NWD_IOU_RATIO:.2f}*(1-CIoU) + "
          f"{1 - NWD_IOU_RATIO:.2f}*(1-NWD), C={NWD_CONSTANT}")
else:
    _ul_loss.BboxLoss.forward = _ORIG_BBOX_FORWARD
    print("NWD patch NOT applied -- running the pure V6 CIoU baseline.")

# --- self-test: identical boxes -> NWD 1.0; shifted boxes -> < 1.0 ---
_a = torch.tensor([[10.0, 10.0, 14.0, 14.0]])
_c2 = torch.tensor([[12.0, 12.0, 16.0, 16.0]])
print("self-test NWD(identical) =", round(float(_nwd_similarity(_a, _a, NWD_CONSTANT)), 4))
print("self-test NWD(shifted 2px) =", round(float(_nwd_similarity(_a, _c2, NWD_CONSTANT)), 4))

## 8. Runtime dataset YAML (full image)

Writes a YAML with **absolute** train/val paths so Ultralytics trains on the original full-image data with no path ambiguity. This is the V6 data — NOT tiled.

In [ ]:
runtime_yaml = Path("/kaggle/working/yolo_seg_v15_nwd.yaml")
yaml_body = {
    "path": str(yaml_root),
    "train": str(train_path),
    "val": str(val_path),
    "names": names,
}
with open(runtime_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(yaml_body, f, sort_keys=False, allow_unicode=True)

YOLO_DATA_YAML = runtime_yaml
print("Runtime dataset YAML:", YOLO_DATA_YAML)
print("-" * 60)
print(runtime_yaml.read_text(encoding="utf-8"))
print("-" * 60)

## 9. Train (V15 NWD)

Stock `yolov8s-seg` + the NWD-patched box loss, full-image V6 augmentation. The validation Mask mAP50-95 reported here is **directly comparable to V6 0.234**.

In [ ]:
# =========================================================
# V15: stock yolov8s-seg + NWD-patched box loss, full-image training
# =========================================================
# Single variable vs V6: the ONLY change is the NWD-blended box regression loss (section 7).
model = YOLO(MODEL_NAME)
print("Built model:", MODEL_NAME)
print("NWD active:", bool(NWD_ENABLE and NWD_IOU_RATIO < 1.0))


def on_train_epoch_start(trainer):
    print(f"\n========== Epoch {trainer.epoch + 1}/{trainer.epochs} started ==========")


def on_fit_epoch_end(trainer):
    print(f"---------- Epoch {trainer.epoch + 1}/{trainer.epochs} finished ----------")
    if getattr(trainer, "metrics", None):
        keys = ["metrics/precision(M)", "metrics/recall(M)",
                "metrics/mAP50(M)", "metrics/mAP50-95(M)"]
        txt = ", ".join(f"{k}: {trainer.metrics[k]:.4f}" for k in keys if k in trainer.metrics)
        if txt:
            print("Val (full-image, directly comparable to the 0.234 V6 baseline):", txt)


model.add_callback("on_train_epoch_start", on_train_epoch_start)
model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

results = model.train(
    data=str(YOLO_DATA_YAML),
    task="segment",
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    seed=SEED,
    patience=PATIENCE,
    cache=False,
    pretrained=True,
    optimizer="auto",
    verbose=False,
    plots=False,
    fliplr=FLIPLR,
    mosaic=MOSAIC,
    close_mosaic=CLOSE_MOSAIC,
    mixup=MIXUP,
    copy_paste=COPY_PASTE,
)

RUN_DIR = Path(model.trainer.save_dir)
WEIGHTS_DIR = RUN_DIR / "weights"
BEST_PT = WEIGHTS_DIR / "best.pt"
LAST_PT = WEIGHTS_DIR / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

print("\nRun directory:", RUN_DIR)
print("best.pt exists:", BEST_PT.exists())
print("results.csv exists:", RESULTS_CSV.exists())
print("\nUse best.pt for evaluation/submission. last.pt is kept only for debugging.")

## 10. Check Training Outputs

In [ ]:
if "RUN_DIR" in globals() and Path(RUN_DIR).exists():
    RUN_DIR = Path(RUN_DIR)
else:
    candidate_run_dirs = sorted(
        [p for p in PROJECT_DIR.glob(f"{RUN_NAME}*") if p.is_dir()],
        key=lambda p: p.stat().st_mtime, reverse=True,
    )
    RUN_DIR = next((p for p in candidate_run_dirs if (p / "weights").exists()), None)
    if RUN_DIR is None:
        raise FileNotFoundError("No YOLO run directory with a weights folder was found.")

WEIGHTS_DIR = RUN_DIR / "weights"
BEST_PT = WEIGHTS_DIR / "best.pt"
LAST_PT = WEIGHTS_DIR / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"

required = {"best.pt": BEST_PT, "last.pt": LAST_PT, "results.csv": RESULTS_CSV}
print("Run directory:", RUN_DIR)
missing = []
for name_, path_ in required.items():
    ok = path_.exists()
    print(f"{name_}: {path_} | exists: {ok}")
    if not ok:
        missing.append(name_)
if missing:
    raise FileNotFoundError(f"Missing required output files: {missing}")

print("\nTraining-only notebook completed successfully.")
print("Save RESULTS_CSV as results/version15_results.csv in the repo.")
print("Main checkpoint:", BEST_PT)

In [ ]:
# Last few rows of the training metrics CSV (no plots / images).
metrics_df = pd.read_csv(RESULTS_CSV)
metrics_df.columns = [c.strip() for c in metrics_df.columns]
cols = [c for c in metrics_df.columns
        if c in ("epoch", "metrics/precision(M)", "metrics/recall(M)",
                 "metrics/mAP50(M)", "metrics/mAP50-95(M)")]
print("Best Mask mAP50-95(M):", round(float(metrics_df["metrics/mAP50-95(M)"].max()), 4),
      "(V6 baseline ~0.234)")
metrics_df[cols].tail(10) if cols else metrics_df.tail(10)

## Next Step

1. Save this run's `results.csv` → `results/version15_results.csv` (the durable record).
2. Diagnose with `tools/val_native_yolo_seg.py` on `best.pt`, and — crucially — **re-run `src/05`**
   to get the small-Caries **localization recall@IoU0.5** vs V6: the most direct box-quality proxy.
3. If Mask mAP50-95 ≥ V6 + 0.003 (and the large classes do not regress) → NWD helped. Then:
   - sweep `NWD_CONSTANT` `{3, 5, 8}` and `NWD_IOU_RATIO` `{0.5, 0.7}` (one variable each);
   - optionally stack **RFLA** (assigner-level) or feed the tighter boxes back into the MedSAM
     refine (`src/07`) to revisit that oracle ceiling.
4. Log a V15 section in `docs/AlphaDent_training_summary_EN.md` / `_CN.md` + README + project memory.

> If the run shows no movement, the first knob to try is **`NWD_CONSTANT`** (it is unit-sensitive),
> then `NWD_IOU_RATIO`. Judge NWD only after a small sweep, not from a single C.